In [21]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
from scrapper import fetch_website_links,fetch_website
from IPython.display import display,Markdown,update_display


In [2]:
load_dotenv(override=True)

True

In [3]:
gemini_api_key=os.getenv('GEMINI_API_KEY')
if gemini_api_key:
    print("API key loaded successfully")
else:
    print("Failed to load API key")

API key loaded successfully


In [9]:
gemini=OpenAI(api_key=gemini_api_key,base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

In [4]:
links=fetch_website_links("https://www.w3schools.com/")

In [5]:
links

['https://www.w3schools.com',
 'https://profile.w3schools.com/log-in',
 'https://campus.w3schools.com/collections/course-catalog',
 'https://order.w3schools.com/plans',
 'https://www.w3schools.com/academy/index.php',
 'https://www.w3schools.com/spaces/index.php',
 'https://www.w3schools.com/practice/index.php',
 'https://spaces.w3schools.com',
 'https://pathfinder.w3schools.com',
 'https://profile.w3schools.com/logout',
 'https://www.youtube.com/@w3schools',
 'https://www.linkedin.com/company/w3schools.com/',
 'https://discord.com/invite/w3schools',
 'https://www.facebook.com/w3schoolscom/',
 'https://www.instagram.com/w3schools.com_official/',
 'https://profile.w3schools.com/signup',
 'https://campus.w3schools.com/collections/course-catalog/products/html-course',
 'https://campus.w3schools.com/collections/course-catalog/products/css-course',
 'https://campus.w3schools.com/collections/course-catalog/products/javascript-course',
 'https://campus.w3schools.com/collections/course-catalog/

In [6]:
link_system_prompt="""
you are provided with list of links of a webpage.
you are able to decide which of the links are relevant to include in a brochure about the company,
such as links to an about page, carrer pages.
You shiuld respond in json as in this example
{
"links": [
    {"type":"about page" , "url":"https://full.url/goes/here/about"},
    {"type":"careers page" , "url":"https://full.url/goes/here/career"}]
}
 """

In [7]:
def get_links_user_prompt(url):
    user_prompt="""
Here is a list of links on the website {url}
Please decide which of these links are relevant for a brochure about the company,
respond with full https URL in json format.
Do not include Terms of Service,Privacy policy, email links.

links:

"""
    links=fetch_website_links(url)
    user_prompt+="\n".join(links)
    return user_prompt

In [8]:
get_links_user_prompt("https://www.w3schools.com/")

'\nHere is a list of links on the website {url}\nPlease decide which of these links are relevant for a brochure about the company,\nrespond with full https URL in json format.\nDo not include Terms of Service,Privacy policy, email links.\n\nlinks:\n\nhttps://www.w3schools.com\nhttps://profile.w3schools.com/log-in\nhttps://campus.w3schools.com/collections/course-catalog\nhttps://order.w3schools.com/plans\nhttps://www.w3schools.com/academy/index.php\nhttps://www.w3schools.com/spaces/index.php\nhttps://www.w3schools.com/practice/index.php\nhttps://spaces.w3schools.com\nhttps://pathfinder.w3schools.com\nhttps://profile.w3schools.com/logout\nhttps://www.youtube.com/@w3schools\nhttps://www.linkedin.com/company/w3schools.com/\nhttps://discord.com/invite/w3schools\nhttps://www.facebook.com/w3schoolscom/\nhttps://www.instagram.com/w3schools.com_official/\nhttps://profile.w3schools.com/signup\nhttps://campus.w3schools.com/collections/course-catalog/products/html-course\nhttps://campus.w3schools.

In [14]:
def select_relevant_links(url):
    response=gemini.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {"role":"system","content":link_system_prompt},
            {"role": "user","content":get_links_user_prompt(url)},
        ],
        response_format={"type":"json_object"}
        )
    result=response.choices[0].message.content
    links=json.loads(result)
    return links

In [15]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'business solutions', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'research', 'url': 'https://huggingface.co/papers'},
  {'type': 'learning resources', 'url': 'https://huggingface.co/learn'},
  {'type': 'resources', 'url': 'https://huggingface.co/docs'},
  {'type': 'support', 'url': 'https://huggingface.co/support'},
  {'type': 'brand assets', 'url': 'https://huggingface.co/brand'},
  {'type': 'product offering', 'url': 'https://huggingface.co/models'},
  {'type': 'product offering', 'url': 'https://huggingface.co/datasets'},
  {'type': 'product offering', 'url': 'https://huggingface.co/spaces'},
  {'type': 'product offering', 'url

In [16]:
def fetch_page_and_all_relevant_links(url):
    contents=fetch_website(url)
    relevant_links=select_relevant_links(url)
    result=f"## Landing page:\n\n{contents}\n##Relevant links:\n"
    for link in relevant_links['links']:
        result+=f"\n\n###Link:{link['type']}\n"
        result+=fetch_website(link["url"])
    return result

In [17]:
fetch_page_and_all_relevant_links("https://huggingface.co")

'## Landing page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nnvidia/LocateAnything-3B\nUpdated\n7 days ago\n•\n78.9k\n•\n1.04k\nLiquidAI/LFM2.5-8B-A1B\nUpdated\n3 days ago\n•\n60.2k\n•\n455\nopenbmb/MiniCPM5-1B\nUpdated\n8 days ago\n•\n68.5k\n•\n738\nHauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive\nUpdated\nApr 17\n•\n2.6M\n•\n1.31k\nstepfun-ai/Step-3.7-Flash\nUpdated\nabout 5 hours ago\n•\n18k\n•

In [18]:
brochure_system_prompt="""
You are an assistant that analyses the contents of several relevant pages from company website and create a short brochure  about the company for propective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of campany culture,customers and careers jobs if you have the information."""

In [19]:
def get_brochure_user_prompt(company_name,url):
    user_prompt=f"""
you are looking at a company called: {company_name}.
Here are the contents of its landing page and other relevant  pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n"""
    user_prompt+=fetch_page_and_all_relevant_links(url)
    user_prompt+=user_prompt[:5_000]
    return user_prompt

In [22]:
def create_brochure(company_name,url):
    response=gemini.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {"role":"system","content":brochure_system_prompt},
            {"role":"user","content":get_brochure_user_prompt(company_name,url)}
        ]
    )
    result=response.choices[0].message.content
    display(Markdown(result))

In [24]:
create_brochure("HuggingFace","https://huggingface.co")

Hugging Face: The AI Community Building the Future

Hugging Face stands at the forefront of the artificial intelligence revolution, serving as the essential collaboration platform for the global machine learning community. We empower developers, researchers, and enterprises to create, discover, and collaborate on cutting-edge AI models, datasets, and applications. Our mission is to build an open and ethical AI future, together.

**For Prospective Customers: Unleash Your AI Potential**

Hugging Face provides an unparalleled ecosystem for all your machine learning needs:

*   **The Hub:** Access and share over 2 million models, 1 million applications (Spaces), and 500,000 datasets. From state-of-the-art text generation and image processing to advanced audio analysis and multimodal tasks, our vast repository supports over 50 ML tasks and a multitude of languages, fostering rapid innovation.
*   **AI Tools & Libraries:** Leverage powerful open-source libraries like Transformers and Diffusers, alongside dedicated tools for tokenization, evaluation, and efficient deployment. Chat with diverse AI models instantly via HuggingChat, our chat app powered by open-source AI.
*   **Enterprise Solutions:** Accelerate your AI initiatives with Hugging Face PRO and Enterprise Support. Benefit from priority assistance, secure Single Sign-On (SSO), robust audit logs, granular access control, and region-specific data management. Our expert team provides guidance on everything from content generation, document AI, and translation models to optimizing inference speed and pre-training.

**For Investors: Powering the Open AI Ecosystem**

Hugging Face represents a strategic investment in the foundational infrastructure of open-source AI. We are the central nervous system for collaborative machine learning, demonstrating:

*   **Market Leadership:** With a rapidly growing community, millions of accessible models, datasets, and applications, we are the de facto platform for open AI development.
*   **Innovation Engine:** Our platform and core libraries drive advancements across diverse ML domains, supported by a talented science team pushing the boundaries of technology.
*   **Enterprise Adoption:** Leading companies and organizations rely on our secure and scalable solutions to integrate AI into their operations, making Hugging Face critical to their growth and efficiency.

**For Recruits: Shape the Future of AI with Us**

Join a vibrant and dynamic team dedicated to an open and ethical AI future. At Hugging Face, you'll find:

*   **Company Culture:** We are a community-driven organization committed to open science, collaboration, and ethical AI development. Your work will directly contribute to fostering an inclusive and innovative environment where sharing and learning are paramount.
*   **Learning & Development:** Grow your skills with access to comprehensive learning resources, including specialized courses on Large Language Models, AI Agents, Robotics, Deep Reinforcement Learning, Computer Vision, and more. Our Open-Source AI Cookbook provides practical, hands-on experience.
*   **Career Opportunities:** We are always looking for passionate individuals to join our mission. Explore current openings on our careers page and become part of a team that's building the next generation of machine learning.

**Join the Future of AI**

Whether you're looking to build, invest, or innovate, Hugging Face offers the platform, community, and expertise to achieve your goals. Connect with us and become a part of the AI community building the future.

Learn more at huggingface.co